# Giustificazioni verificabili e ritiro su LUBM

Benchmark con **numero di query configurabile**. Impostare `N_QUERIES = "all"` per tutte
le query ammissibili oppure un intero positivo, per esempio `25`.
Le query positive provengono da `owlapy_generated_qa.json`; HermiT verifica l'entailment
originale e il non-entailment dopo il ritiro di un singolo assioma.

Il builder verifica tutto il pool prima di selezionare N casi senza ripetizioni, con seed fisso.
Non impone quote per profondità o soggetti distinti. Se N supera i casi disponibili, dà un errore
esplicito senza produrre un dataset ridotto. Ogni nuovo dataset è congelato in una directory propria;
il dataset `subclass_grounding_v1` da sei casi rimane invariato.

Ogni query comporta due chiamate: **intera ontologia originale**, poi **intera ontologia aggiornata**,
con la risposta autentica precedente nella cronologia. Le coppie sono indipendenti.
Il costo è `2 × N × numero di modelli` chiamate. La selezione e i controlli HermiT non chiamano Ollama.

## Ambiente e avvio

Java nel PATH, Ollama avviato e un modello già installato. Kernel **Python (epistemic pilot)**.

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-belief-revision-full.txt
```

Configurare la prossima cella. Eseguire solo le celle prima del ciclo su `MODELS` per preparare
e verificare il dataset; **Run All** esegue anche il benchmark LLM. Nessun modello viene scaricato.
Gli output del precedente pilot restano nel run storico
`generated/subclass_grounding/20260929T125224Z_cbd1a24f/`; sono stati rimossi dalle celle aggiornate
per non confonderli con il nuovo dataset.


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
ROOT = next((p.resolve() for p in roots if (p / "subclass_grounding_benchmark.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from subclass_grounding_benchmark import OPTIONS, build_dataset, load_dataset, run_benchmark, compare_runs, message

MODELS = ["qwen2.5-coder:7b"]  # aggiungere solo nomi esatti di modelli locali
N_QUERIES = "all"  # oppure, per esempio, 25
SELECTION_SEED = 20260929
suffix = "" if SELECTION_SEED == 20260929 else f"_seed{SELECTION_SEED}"
DATASET_DIR = ROOT / "benchmarks" / f"subclass_grounding_v2_{N_QUERIES}{suffix}"
if not DATASET_DIR.exists():
    build_dataset(DATASET_DIR, seed=SELECTION_SEED, n_queries=N_QUERIES)
data, graphs, texts, oracle, dataset_hash = load_dataset(DATASET_DIR)
if data.get("requested_n_queries") != N_QUERIES or data["seed"] != SELECTION_SEED:
    raise ValueError("Il dataset esistente ha una configurazione diversa. Usare una directory nuova.")
print("Casi ammissibili:", data["eligible_queries"], "| selezionati:", len(data["cases"]))
print("Distribuzione per profondità:", data["selected_by_depth"])
print("Chiamate previste:", 2 * len(data["cases"]) * len(MODELS))
print("Dataset:", data["version"], "| SHA-256:", dataset_hash)
print("Casi:", len(data["cases"]), "| opzioni fisse:", OPTIONS)
table = "| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |\n|---|---|---|---|"
for case in data["cases"]:
    short = lambda uri: uri.split("#")[-1]
    removed = case["removed_axioms"][0]
    table += f"\n| {case['id']} | {short(case['query'][0])} ⊑ {short(case['query'][1])} | {case['explicit_proof_depth']} | {short(removed['subject'])} ⊑ {short(removed['object'])} |"
display(Markdown(table))

Casi ammissibili: 52 | selezionati: 52
Distribuzione per profondità: {'1': 33, '2': 15, '3': 4}
Chiamate previste: 104
Dataset: subclass_grounding_v2_all | SHA-256: 8710b0ce845545514c6aba52a076062c48982e1802a863463457e034faf9db3b
Casi: 52 | opzioni fisse: {'temperature': 0, 'seed': 42, 'num_ctx': 32768, 'num_predict': 768}


| Caso | Query | Passaggi nella prova esplicita | Assioma ritirato |
|---|---|---|---|
| subclass_01 | AdministrativeStaff ⊑ Employee | 1 | AdministrativeStaff ⊑ Employee |
| subclass_02 | Article ⊑ Publication | 1 | Article ⊑ Publication |
| subclass_03 | AssistantProfessor ⊑ Professor | 1 | AssistantProfessor ⊑ Professor |
| subclass_04 | AssociateProfessor ⊑ Professor | 1 | AssociateProfessor ⊑ Professor |
| subclass_05 | Book ⊑ Publication | 1 | Book ⊑ Publication |
| subclass_06 | Chair ⊑ Professor | 1 | Chair ⊑ Professor |
| subclass_07 | ClericalStaff ⊑ AdministrativeStaff | 1 | ClericalStaff ⊑ AdministrativeStaff |
| subclass_08 | College ⊑ Organization | 1 | College ⊑ Organization |
| subclass_09 | ConferencePaper ⊑ Article | 1 | ConferencePaper ⊑ Article |
| subclass_10 | Course ⊑ Work | 1 | Course ⊑ Work |
| subclass_11 | Dean ⊑ Professor | 1 | Dean ⊑ Professor |
| subclass_12 | Department ⊑ Organization | 1 | Department ⊑ Organization |
| subclass_13 | Faculty ⊑ Employee | 1 | Faculty ⊑ Employee |
| subclass_14 | FullProfessor ⊑ Professor | 1 | FullProfessor ⊑ Professor |
| subclass_15 | GraduateCourse ⊑ Course | 1 | GraduateCourse ⊑ Course |
| subclass_16 | GraduateStudent ⊑ Person | 1 | GraduateStudent ⊑ Person |
| subclass_17 | Institute ⊑ Organization | 1 | Institute ⊑ Organization |
| subclass_18 | JournalArticle ⊑ Article | 1 | JournalArticle ⊑ Article |
| subclass_19 | Lecturer ⊑ Faculty | 1 | Lecturer ⊑ Faculty |
| subclass_20 | Manual ⊑ Publication | 1 | Manual ⊑ Publication |
| subclass_21 | PostDoc ⊑ Faculty | 1 | PostDoc ⊑ Faculty |
| subclass_22 | Professor ⊑ Faculty | 1 | Professor ⊑ Faculty |
| subclass_23 | Program ⊑ Organization | 1 | Program ⊑ Organization |
| subclass_24 | Research ⊑ Work | 1 | Research ⊑ Work |
| subclass_25 | ResearchGroup ⊑ Organization | 1 | ResearchGroup ⊑ Organization |
| subclass_26 | Software ⊑ Publication | 1 | Software ⊑ Publication |
| subclass_27 | Specification ⊑ Publication | 1 | Specification ⊑ Publication |
| subclass_28 | SystemsStaff ⊑ AdministrativeStaff | 1 | SystemsStaff ⊑ AdministrativeStaff |
| subclass_29 | TechnicalReport ⊑ Article | 1 | TechnicalReport ⊑ Article |
| subclass_30 | UndergraduateStudent ⊑ Student | 1 | UndergraduateStudent ⊑ Student |
| subclass_31 | University ⊑ Organization | 1 | University ⊑ Organization |
| subclass_32 | UnofficialPublication ⊑ Publication | 1 | UnofficialPublication ⊑ Publication |
| subclass_33 | VisitingProfessor ⊑ Professor | 1 | VisitingProfessor ⊑ Professor |
| subclass_34 | AssistantProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_35 | AssociateProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_36 | Chair ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_37 | ClericalStaff ⊑ Employee | 2 | AdministrativeStaff ⊑ Employee |
| subclass_38 | ConferencePaper ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_39 | Dean ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_40 | FullProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_41 | GraduateCourse ⊑ Work | 2 | Course ⊑ Work |
| subclass_42 | JournalArticle ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_43 | Lecturer ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_44 | PostDoc ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_45 | Professor ⊑ Employee | 2 | Faculty ⊑ Employee |
| subclass_46 | SystemsStaff ⊑ Employee | 2 | AdministrativeStaff ⊑ Employee |
| subclass_47 | TechnicalReport ⊑ Publication | 2 | Article ⊑ Publication |
| subclass_48 | VisitingProfessor ⊑ Faculty | 2 | Professor ⊑ Faculty |
| subclass_49 | AssistantProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_50 | AssociateProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_51 | FullProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |
| subclass_52 | VisitingProfessor ⊑ Employee | 3 | Faculty ⊑ Employee |

## Output richiesto e verifica formale

Il modello restituisce soltanto `answer` (`entailed` / `not_entailed`) e `axioms`:

```json
{
  "answer": "entailed",
  "axioms": [
    {
      "subject": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Course",
      "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf",
      "object": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Work"
    }
  ]
}
```

L'esempio sopra illustra il formato al lettore del notebook; non è inserito nei prompt.
Con `not_entailed`, `axioms` deve essere `[]`.

Per una risposta positiva con insieme di assiomi citati (J), snapshot (O) e query (q),
verifichiamo:

- correttezza della risposta rispetto a HermiT su (O);
- **provenienza:** (J \subseteq O), quindi ogni assioma è realmente asserito nello snapshot;
- **sufficienza:** (J \models q), tramite HermiT sul solo supporto citato;
- minimalità per inclusione, registrata separatamente e non richiesta per superare il caso.

Una prova alternativa valida passa: non facciamo exact match con la prova di riferimento.
Citare una conclusione inferita come se fosse un assioma asserito non passa.

Per una risposta negativa, la lista vuota è una convenzione di output e non una prova:
il non-entailment è verificato sullo snapshot completo. Nessun testo libero viene richiesto,
quindi questo esperimento misura il supporto strutturato dichiarato e non la coerenza di una spiegazione narrativa.

## Score

Per ogni coppia (i):

- (G_i=1) se la risposta originale è corretta e il suo supporto è presente e sufficiente; altrimenti 0.
- (R_i=1) se dopo il ritiro la risposta è `not_entailed` con lista vuota; altrimenti 0.
- (P_i=G_iR_i): successo sull'intera coppia.

Riportiamo **grounded score** (\sum G_i/N), **withdrawal score** (\sum R_i/N),
e **paired grounded revision score** (\sum P_i/N). Quest'ultimo è lo score congiunto principale.
Il denominatore è sempre costituito da tutte le N query selezionate: non escludiamo le baseline sbagliate.
Riportiamo anche l'accuracy iniziale, le risposte iniziali corrette prive di supporto valido
e l'eventuale riuso di assiomi ritirati.

Errori di formato del modello contano come insuccessi. Errori infrastrutturali, di contesto o
chiamate mancanti rendono il run **non confrontabile**, anziché abbassare arbitrariamente lo score.
I dettagli e gli insuccessi restano nei log. La finestra fissa è 32768 token; il budget conservativo
e la capacità riportata da Ollama vengono controllati senza tagliare il contesto.

In [4]:
# Ispezionare questa variabile per vedere l'intero prompt, senza chiamare il modello.
example_request = message(data["cases"][0], "original", texts)
print("Caratteri nel primo messaggio:", len(example_request["content"]))
print("Triple nello snapshot originale:", len(graphs["ontology_original.ttl"]))


Caratteri nel primo messaggio: 11789
Triple nello snapshot originale: 319


## esecuzione


### gemma4:e4b


In [7]:
run_dirs = []
results = {}
model = "gemma4:e4b"
print("\nMODELLO:", model, flush=True)
directory, result = run_benchmark(model, DATASET_DIR)
run_dirs.append(directory)
results[model] = result


MODELLO: gemma4:e4b
subclass_01 original {'answer': 'not_entailed', 'axioms': []}
subclass_01 withdrawal {'answer': 'not_entailed', 'axioms': []}
subclass_02 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article'}]}
subclass_02 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article'}]}
subclass_03 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor'}]}
subclass_03 withdrawal {'answer': 'not_entailed', 'ax

In [8]:
for model, result in results.items():
    display(Markdown(f"### {model}"))
    summary = result["summary"]
    display(Markdown(
        "| Misura | Valore |\n|---|---|\n" + "\n".join(
            f"| {key} | {summary[key]} |" for key in (
                "complete_for_comparison", "paired_grounded_revision_score", "original_grounded_score",
                "withdrawal_score", "original_answer_accuracy", "correct_original_without_valid_support_count"))))
    print("Score per profondità:", json.dumps(result["by_depth"], indent=2))
    table = "| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |\n|---|---|---|---|"
    for pair in result["pairs"]:
        table += f"\n| {pair['case_id']} | {pair['original_success']} | {pair['withdrawal_success']} | {pair['pair_success']} |"
    display(Markdown(table))
    for row in result["responses"]:
        print(row["case_id"], row["condition"], "risposta:", row["answer"],
              "atteso:", row["gold"], "supporto sufficiente:", row["support_sufficient"])
print("Cartelle dei risultati:")
for directory in run_dirs:
    print(directory)

### gemma4:e4b

| Misura | Valore |
|---|---|
| complete_for_comparison | True |
| paired_grounded_revision_score | 0.75 |
| original_grounded_score | 0.8653846153846154 |
| withdrawal_score | 0.8076923076923077 |
| original_answer_accuracy | 0.8653846153846154 |
| correct_original_without_valid_support_count | 0 |

Score per profondità: {
  "1": {
    "n_pairs": 33,
    "original_grounded_score": 0.9696969696969697,
    "withdrawal_score": 0.9696969696969697,
    "paired_grounded_revision_score": 0.9393939393939394
  },
  "2": {
    "n_pairs": 15,
    "original_grounded_score": 0.8,
    "withdrawal_score": 0.6,
    "paired_grounded_revision_score": 0.5333333333333333
  },
  "3": {
    "n_pairs": 4,
    "original_grounded_score": 0.25,
    "withdrawal_score": 0.25,
    "paired_grounded_revision_score": 0.0
  }
}


| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |
|---|---|---|---|
| subclass_01 | False | True | False |
| subclass_02 | True | False | False |
| subclass_03 | True | True | True |
| subclass_04 | True | True | True |
| subclass_05 | True | True | True |
| subclass_06 | True | True | True |
| subclass_07 | True | True | True |
| subclass_08 | True | True | True |
| subclass_09 | True | True | True |
| subclass_10 | True | True | True |
| subclass_11 | True | True | True |
| subclass_12 | True | True | True |
| subclass_13 | True | True | True |
| subclass_14 | True | True | True |
| subclass_15 | True | True | True |
| subclass_16 | True | True | True |
| subclass_17 | True | True | True |
| subclass_18 | True | True | True |
| subclass_19 | True | True | True |
| subclass_20 | True | True | True |
| subclass_21 | True | True | True |
| subclass_22 | True | True | True |
| subclass_23 | True | True | True |
| subclass_24 | True | True | True |
| subclass_25 | True | True | True |
| subclass_26 | True | True | True |
| subclass_27 | True | True | True |
| subclass_28 | True | True | True |
| subclass_29 | True | True | True |
| subclass_30 | True | True | True |
| subclass_31 | True | True | True |
| subclass_32 | True | True | True |
| subclass_33 | True | True | True |
| subclass_34 | True | True | True |
| subclass_35 | True | True | True |
| subclass_36 | True | True | True |
| subclass_37 | False | True | False |
| subclass_38 | True | False | False |
| subclass_39 | True | True | True |
| subclass_40 | True | True | True |
| subclass_41 | True | True | True |
| subclass_42 | True | False | False |
| subclass_43 | True | False | False |
| subclass_44 | True | True | True |
| subclass_45 | False | False | False |
| subclass_46 | False | False | False |
| subclass_47 | True | False | False |
| subclass_48 | True | True | True |
| subclass_49 | False | True | False |
| subclass_50 | False | False | False |
| subclass_51 | True | False | False |
| subclass_52 | False | False | False |

subclass_01 original risposta: not_entailed atteso: entailed supporto sufficiente: None
subclass_01 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_02 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_02 withdrawal risposta: entailed atteso: not_entailed supporto sufficiente: False
subclass_03 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_03 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_04 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_04 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_05 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_05 withdrawal risposta: not_entailed atteso: not_entailed supporto sufficiente: None
subclass_06 original risposta: entailed atteso: entailed supporto sufficiente: True
subclass_06 withdrawal ri

## Confrontare i modelli e leggere i limiti

I risultati includono anche gli score separati per profondità della prova.

`compare_runs` accetta solo run completi con lo stesso fingerprint di dataset, istruzioni, schema,
opzioni ed evaluator. Si possono aggiungere cartelle di run precedenti alla lista della prossima cella.
Il codice aggiornato ha un fingerprint diverso dal pilot storico: rieseguire i modelli con lo stesso codice per nuovi confronti.
Digest del modello, versione Ollama, Java, librerie e piattaforma sono registrati nel manifest.
Per confronti sperimentali controllati mantenere anche ambiente e hardware costanti, o documentare le differenze.

Il numero di query non garantisce da solo una graduatoria affidabile. Le query condividono
parte degli assiomi e alcune ritirano lo stesso collegamento: non sono prove indipendenti.
Le query che richiedono assiomi complessi (restrizioni, equivalenze) senza una prova di sole inclusioni
nominate sono fuori dallo scope di questa prima versione, pur restando quei costrutti nel contesto.

In questo protocollo tutti i secondi turni sono negativi. Un modello che impara semplicemente a
cambiare risposta può quindi ottenere un alto score: serviranno controlli senza variazione di entailment
per una misura più forte della revisione selettiva. Qui manteniamo deliberatamente il primo test semplice
richiesto. Non interpretiamo una prova valida come evidenza del processo interno realmente seguito.

In [9]:
# Aggiungere qui altre cartelle di run per un confronto sul medesimo protocollo.
comparison_dirs = list(run_dirs)
if all(result["summary"]["complete_for_comparison"] for result in results.values()):
    comparison = compare_runs(comparison_dirs)
    for row in comparison:
        print(row["model"], "| coppie:", row["pair_success_count"], "/", row["n_pairs"],
              "| score:", row["paired_grounded_revision_score"], "| digest:", row["digest"])
else:
    print("Almeno un run è incompleto: nessuna graduatoria generata.")

gemma4:e4b | coppie: 39 / 52 | score: 0.75 | digest: c6eb396dbd5992bbe3f5cdb947e8bbc0ee413d7c17e2beaae69f5d569cf982eb


## Run dedicato a gemma3:12b

Questa sezione è autonoma: eseguire **solo le celle da qui in poi** per provare Gemma 3 senza
rilanciare Gemma 4. Usa lo stesso dataset congelato `subclass_grounding_v2_all` (52 query,
104 chiamate) e le stesse opzioni del run `gemma4:e4b` sopra: temperature 0, seed 42,
contesto 32768 token, output massimo 768 token.

Le variabili `gemma3_*` sono separate da `run_dirs` e `results` usate da Gemma 4.
Ogni esecuzione crea una nuova cartella in `generated/subclass_grounding/` e conserva i log
progressivamente. Nessun risultato precedente viene sovrascritto.


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

_gemma3_roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
GEMMA3_ROOT = next((p.resolve() for p in _gemma3_roots if (p / "subclass_grounding_benchmark.py").exists()), None)
if GEMMA3_ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(GEMMA3_ROOT) not in sys.path:
    sys.path.insert(0, str(GEMMA3_ROOT))
from subclass_grounding_benchmark import load_dataset, run_benchmark

GEMMA3_MODEL = "gemma3:12b"
GEMMA3_DATASET_DIR = GEMMA3_ROOT / "benchmarks" / "subclass_grounding_v2_all"
GEMMA3_OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 32768, "num_predict": 768}
GEMMA3_EXPECTED_DATASET_SHA256 = "8710b0ce845545514c6aba52a076062c48982e1802a863463457e034faf9db3b"
gemma3_data, gemma3_graphs, gemma3_texts, gemma3_oracle, gemma3_dataset_hash = load_dataset(GEMMA3_DATASET_DIR)
if gemma3_dataset_hash != GEMMA3_EXPECTED_DATASET_SHA256:
    raise ValueError("Il dataset è diverso da quello del run Gemma 4: verificare il confronto prima di procedere.")
print("Modello:", GEMMA3_MODEL)
print("Dataset:", gemma3_data["version"], "| SHA-256:", gemma3_dataset_hash)
print("Query:", len(gemma3_data["cases"]), "| chiamate:", 2 * len(gemma3_data["cases"]))
print("Opzioni:", GEMMA3_OPTIONS)


Modello: gemma3:12b
Dataset: subclass_grounding_v2_all | SHA-256: 8710b0ce845545514c6aba52a076062c48982e1802a863463457e034faf9db3b
Query: 52 | chiamate: 104
Opzioni: {'temperature': 0, 'seed': 42, 'num_ctx': 32768, 'num_predict': 768}


In [2]:
# Esegue esclusivamente Gemma 3. Il runner assegna un nuovo run_id automaticamente.
gemma3_directory, gemma3_result = run_benchmark(
    GEMMA3_MODEL, GEMMA3_DATASET_DIR, options=GEMMA3_OPTIONS
)
print("Risultati Gemma 3:", gemma3_directory)


subclass_01 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AdministrativeStaff', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Employee'}]}


subclass_01 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AdministrativeStaff', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Employee'}]}


subclass_02 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication'}]}


subclass_02 withdrawal {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Article', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Publication'}]}


subclass_03 original {'answer': 'entailed', 'axioms': [{'object': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor', 'predicate': 'http://www.w3.org/2000/01/rdf-schema#subClassOf', 'subject': 'http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor'}]}


...

subclass_52 original {'answer': 'not_entailed', 'axioms': []}


subclass_52 withdrawal {'answer': 'not_entailed', 'axioms': []}


{
  "n_pairs": 52,
  "n_responses": 104,
  "complete_for_comparison": true,
  "infrastructure_or_context_errors": 0,
  "format_errors": 0,
  "pair_success_count": 3,
  "paired_grounded_revision_score": 0.057692307692307696,
  "original_grounded_score": 0.057692307692307696,
  "withdrawal_score": 0.6153846153846154,
  "original_answer_accuracy": 0.8461538461538461,
  "correct_original_without_valid_support_count": 41,
  "withdrawal_stale_citation_count": 0
}


Run: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260930T094649Z_1002ba10


Risultati Gemma 3: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260930T094649Z_1002ba10


In [3]:
display(Markdown("### Risultati gemma3:12b"))
gemma3_summary = gemma3_result["summary"]
gemma3_metric_keys = (
    "complete_for_comparison", "n_pairs", "n_responses", "pair_success_count",
    "paired_grounded_revision_score", "original_grounded_score", "withdrawal_score",
    "original_answer_accuracy", "correct_original_without_valid_support_count",
    "withdrawal_stale_citation_count", "format_errors", "infrastructure_or_context_errors",
)
display(Markdown("| Misura | Valore |\n|---|---|\n" + "\n".join(
    f"| {key} | {gemma3_summary[key]} |" for key in gemma3_metric_keys)))
print("Score per profondità:", json.dumps(gemma3_result["by_depth"], indent=2))
gemma3_table = "| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |\n|---|---|---|---|"
for gemma3_pair in gemma3_result["pairs"]:
    gemma3_table += f"\n| {gemma3_pair['case_id']} | {gemma3_pair['original_success']} | {gemma3_pair['withdrawal_success']} | {gemma3_pair['pair_success']} |"
display(Markdown(gemma3_table))
print("Manifest:", gemma3_directory / "manifest.json")
print("Risposte grezze:", gemma3_directory / "responses.jsonl")
print("Valutazione:", gemma3_directory / "evaluation.json")
if not gemma3_summary["complete_for_comparison"]:
    print("Run incompleto per problemi infrastrutturali o di contesto: non usare gli score per una graduatoria.")


### Risultati gemma3:12b

| Misura | Valore |
|---|---|
| complete_for_comparison | True |
| n_pairs | 52 |
| n_responses | 104 |
| pair_success_count | 3 |
| paired_grounded_revision_score | 0.057692307692307696 |
| original_grounded_score | 0.057692307692307696 |
| withdrawal_score | 0.6153846153846154 |
| original_answer_accuracy | 0.8461538461538461 |
| correct_original_without_valid_support_count | 41 |
| withdrawal_stale_citation_count | 0 |
| format_errors | 0 |
| infrastructure_or_context_errors | 0 |

Score per profondità: {
  "1": {
    "n_pairs": 33,
    "original_grounded_score": 0.09090909090909091,
    "withdrawal_score": 0.8181818181818182,
    "paired_grounded_revision_score": 0.09090909090909091
  },
  "2": {
    "n_pairs": 15,
    "original_grounded_score": 0.0,
    "withdrawal_score": 0.2,
    "paired_grounded_revision_score": 0.0
  },
  "3": {
    "n_pairs": 4,
    "original_grounded_score": 0.0,
    "withdrawal_score": 0.5,
    "paired_grounded_revision_score": 0.0
  }
}


| Caso | Originale giustificato | Ritiro corretto | Coppia riuscita |
|---|---|---|---|
| subclass_01 | False | False | False |
| subclass_02 | False | False | False |
| subclass_03 | False | True | False |
| subclass_04 | False | True | False |
| subclass_05 | False | True | False |
| subclass_06 | False | True | False |
| subclass_07 | True | True | True |
| subclass_08 | False | False | False |
| subclass_09 | False | True | False |
| subclass_10 | False | True | False |
| subclass_11 | False | True | False |
| subclass_12 | False | True | False |
| subclass_13 | False | True | False |
| subclass_14 | False | True | False |
| subclass_15 | True | True | True |
| subclass_16 | False | False | False |
| subclass_17 | False | True | False |
| subclass_18 | False | True | False |
| subclass_19 | False | True | False |
| subclass_20 | False | True | False |
| subclass_21 | False | True | False |
| subclass_22 | True | True | True |
| subclass_23 | False | True | False |
| subclass_24 | False | True | False |
| subclass_25 | False | True | False |
| subclass_26 | False | True | False |
| subclass_27 | False | False | False |
| subclass_28 | False | True | False |
| subclass_29 | False | True | False |
| subclass_30 | False | False | False |
| subclass_31 | False | True | False |
| subclass_32 | False | True | False |
| subclass_33 | False | True | False |
| subclass_34 | False | False | False |
| subclass_35 | False | False | False |
| subclass_36 | False | False | False |
| subclass_37 | False | False | False |
| subclass_38 | False | False | False |
| subclass_39 | False | False | False |
| subclass_40 | False | False | False |
| subclass_41 | False | True | False |
| subclass_42 | False | False | False |
| subclass_43 | False | True | False |
| subclass_44 | False | False | False |
| subclass_45 | False | False | False |
| subclass_46 | False | False | False |
| subclass_47 | False | True | False |
| subclass_48 | False | False | False |
| subclass_49 | False | True | False |
| subclass_50 | False | False | False |
| subclass_51 | False | False | False |
| subclass_52 | False | True | False |

Manifest: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260930T094649Z_1002ba10\manifest.json
Risposte grezze: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260930T094649Z_1002ba10\responses.jsonl
Valutazione: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\subclass_grounding\20260930T094649Z_1002ba10\evaluation.json


Le tabelle di Gemma 4 sopra conservano il run precedente. Il confronto va letto sullo stesso
dataset e protocollo; digest del modello, versione Ollama, hardware e dipendenze del nuovo run
sono registrati nel manifest. Per un confronto automatico tramite `compare_runs` occorrono
anche i file del run Gemma 4 sul computer locale: i soli output nel notebook non ne sostituiscono
il manifest. Il report di Gemma 3 distingue errori di formato da errori infrastrutturali.


## Confronto e valutazione: Gemma 4 e Gemma 3


### Come leggere le metriche

Ogni query forma una **coppia**: una risposta sull'ontologia originale e una dopo il ritiro
di un assioma. Indichiamo con **N** il numero di query (qui 52): sono quindi previste **2N**
risposte (104). Tutte le query selezionate restano nel denominatore, anche quando il modello
sbaglia già la risposta iniziale.

Per ciascuna query i definiamo due indicatori, che valgono 0 oppure 1:

- **Gᵢ = 1 (successo originale)**: la risposta iniziale è corretta (`entailed`) e gli assiomi
  citati sono tutti presenti nell'ontologia originale e sufficienti a dimostrare la query,
  secondo HermiT. Una prova alternativa valida è accettata: non serve riprodurre quella di riferimento.
- **Rᵢ = 1 (successo dopo il ritiro)**: la risposta aggiornata è corretta (`not_entailed`)
  e la lista degli assiomi è vuota. Questo vale per il dataset attuale, in cui tutti i ritiri
  rendono la query non più implicata. La lista vuota è una convenzione di output, non una prova
  di non-entailment: quest'ultimo è verificato da HermiT sull'ontologia aggiornata completa.

| Metrica nel file | Calcolo e significato | Lettura |
|---|---|---|
| `paired_grounded_revision_score` | Σ(Gᵢ × Rᵢ) / N: quota di query per cui riescono **entrambi** i turni della stessa coppia. È lo score principale. | Più alto è meglio. Non è la media né il prodotto dei due score aggregati. |
| `original_grounded_score` | ΣGᵢ / N: quota di risposte iniziali corrette con supporto presente e sufficiente. | Misura la correttezza accompagnata da una giustificazione verificabile. |
| `withdrawal_score` | ΣRᵢ / N: quota di risposte corrette dopo il ritiro, con lista vuota. | Include anche le query con risposta iniziale sbagliata; non è condizionato al successo originale. |
| `original_answer_accuracy` | Numero di risposte iniziali con etichetta corretta / N, indipendentemente dalla validità del supporto. | Un valore alto può coesistere con prove errate o assenti. Output non interpretabili non contano come risposte corrette. |
| `pair_success_count` | Σ(Gᵢ × Rᵢ): numero assoluto di coppie riuscite. | Diviso per `n_pairs` produce lo score congiunto. |
| `correct_original_without_valid_support_count` | Numero di risposte iniziali con etichetta corretta ma Gᵢ = 0. | Evidenzia il divario tra risposta corretta e supporto valido; più basso è meglio. |
| `withdrawal_stale_citation_count` | Numero di risposte dopo il ritiro che citano almeno un assioma presente nell'ontologia originale ma assente in quella aggiornata. | Conta le **risposte**, non il numero di assiomi obsoleti. Zero non garantisce che le risposte siano corrette. |
| `format_errors` | Numero di record con errore di formato: per esempio JSON non valido, violazione dello schema o generazione arrivata al limite di output. | Sono insuccessi del modello e restano nel calcolo; il conteggio riguarda entrambi i turni, quindi fino a 2N record. |
| `infrastructure_or_context_errors` | Numero di record con errore infrastrutturale, di contesto o di dipendenza da una prima chiamata inutilizzabile. | Un guasto iniziale può generare anche un errore di dipendenza al secondo turno: non è il conteggio delle sole cause originarie. |
| `complete_for_comparison` | Vero quando non ci sono errori infrastrutturali, di contesto o di dipendenza. | Può essere vero anche con errori di formato. Non certifica da solo la compatibilità tra due run, che richiede anche il controllo dei fingerprint. |
| `n_pairs` / `n_responses` | Numero di coppie valutate / numero di record di risposta valutati. | I record possono includere errori o chiamate saltate: `n_responses` non implica altrettante chiamate LLM riuscite. |

**Esempio dei run storici:** Gemma 4 supera 45 originali su 52 (86,54%) e 42 ritiri
su 52 (80,77%), ma solo 39 query superano entrambi i turni: lo score congiunto è 39/52 = 75%.
Gemma 3 ha 44 etichette iniziali corrette, ma solo 3 supporti validi: accuracy 84,62%,
grounded score 5,77% e 41 risposte corrette senza supporto valido.

Gli score **per profondità** applicano le stesse formule al solo gruppo di query con quella
profondità della prova di riferimento: i denominatori sono 33, 15 e 4, rispettivamente.
La profondità non è il numero di assiomi citati dal modello.

Nei dettagli delle risposte, `citations_current` indica che tutti gli assiomi citati sono
presenti nello snapshot; `support_sufficient` verifica il supporto positivo con HermiT
(ed è falso se ci sono assiomi assenti); `support_minimal` indica che togliendo un qualsiasi
assioma dal supporto sufficiente la prova non regge più. La minimalità è registrata ma
**non è richiesta** per ottenere Gᵢ = 1. Per risposte negative o non interpretabili alcune
verifiche possono essere `null`: ciò non equivale a una verifica superata.

Se il run ha errori infrastrutturali o di contesto, gli score aggregati e per profondità
sono `null` e `complete_for_comparison` è falso; i conteggi e i dettagli restano disponibili
per la diagnosi. Gli errori di formato, invece, abbassano gli score senza rendere il run incompleto.


### Riepilogo dei run già eseguiti

Questa tabella trascrive gli output salvati sopra nel notebook: è un riepilogo storico,
non il risultato di una nuova esecuzione né una verifica dei manifest.
Entrambi riportano 52 query e 104 risposte sul dataset `subclass_grounding_v2_all`.

| Misura | gemma4:e4b | gemma3:12b |
|---|---:|---:|
| Coppie riuscite | 39 / 52 | 3 / 52 |
| Paired grounded revision score | 75,00% | 5,77% |
| Original grounded score | 86,54% | 5,77% |
| Withdrawal score | 80,77% | 61,54% |
| Accuracy della risposta originale | 86,54% | 84,62% |
| Originali corretti senza supporto valido | 0 | 41 |
| Errori di formato (su 104 risposte) | 11 | 0 |
| Risposte di ritiro con citazioni obsolete | 4 | 0 |
| Errori infrastrutturali o di contesto | 0 | 0 |

L'accuracy iniziale è simile, ma la qualità del supporto dichiarato è molto diversa:
Gemma 3 dà 44 risposte iniziali corrette, delle quali 41 non hanno un supporto valido.
Gemma 4 ne dà 45 corrette e tutte hanno supporto valido. Il divario sullo score congiunto
è di **69,23 punti percentuali** a favore di Gemma 4 in questi run.
L'assenza di errori di formato in Gemma 3 indica conformità al formato, non validità della prova.

| Profondità della prova | Query | Coppie riuscite Gemma 4 | Coppie riuscite Gemma 3 |
|---|---:|---:|---:|
| 1 | 33 | 31 / 33 (93,94%) | 3 / 33 (9,09%) |
| 2 | 15 | 8 / 15 (53,33%) | 0 / 15 (0,00%) |
| 3 | 4 | 0 / 4 (0,00%) | 0 / 4 (0,00%) |

Gemma 4 perde efficacia con l'aumento della profondità; nessuno dei due supera una coppia
alla profondità 3, che comprende però solo quattro casi. Le query condividono assiomi e
non sono osservazioni indipendenti; questi risultati non dimostrano una superiorità generale.
Inoltre tutti i turni dopo il ritiro hanno risposta attesa negativa: il withdrawal score
non misura da solo la capacità di distinguere quando una credenza debba essere mantenuta.

### Confronto verificato dai file

Le celle seguenti si possono eseguire autonomamente e **non chiamano Ollama**.
I percorsi selezionano esplicitamente i due run storici: modificarli per confrontare nuovi run.
Se una cartella manca, viene indicato cosa copiare dal computer che ha eseguito il benchmark.
Solo con entrambi i run disponibili si verificano completezza e fingerprint del protocollo.
Il fingerprint include l'intero file Python dell'evaluator: anche la modifica al nome delle
cartelle lo cambia; run vecchi e nuovi possono quindi essere rifiutati dal controllo stretto.


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown

comparison_roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
comparison_root = next((p.resolve() for p in comparison_roots
                        if (p / "subclass_grounding_benchmark.py").exists()), None)
if comparison_root is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
comparison_run_dirs = {
    "gemma4:e4b": comparison_root / "generated/subclass_grounding/20260929T152133Z_ae9c9847",
    "gemma3:12b": comparison_root / "generated/subclass_grounding/20260930T094649Z_1002ba10",
}
comparison_loaded = {}
comparison_ready = False
comparison_missing = [directory / filename
                      for directory in comparison_run_dirs.values()
                      for filename in ("manifest.json", "evaluation.json")
                      if not (directory / filename).is_file()]
if comparison_missing:
    print("Confronto da file non disponibile. File mancanti:")
    for missing in comparison_missing:
        print(missing)
    print("Copiare le cartelle dei run dal computer di origine e rieseguire questa cella.")
    print("Il riepilogo storico sopra resta consultabile, ma i manifest non sono stati verificati.")
else:
    for model_name, directory in comparison_run_dirs.items():
        manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
        evaluation = json.loads((directory / "evaluation.json").read_text(encoding="utf-8"))
        if manifest["model"]["name"] != model_name:
            raise ValueError(f"Modello inatteso nella cartella {directory}")
        comparison_loaded[model_name] = {"manifest": manifest, "evaluation": evaluation}
    fingerprints = {run["manifest"]["protocol_fingerprint"] for run in comparison_loaded.values()}
    if len(fingerprints) != 1:
        print("Confronto bloccato: fingerprint diversi (dataset, prompt, schema, opzioni o codice evaluator).")
    elif not all(run["evaluation"]["summary"]["complete_for_comparison"]
                 for run in comparison_loaded.values()):
        print("Confronto bloccato: almeno un run è incompleto per errori infrastrutturali o di contesto.")
    else:
        comparison_ready = True
        print("Run completi e fingerprint coincidenti. Confronto disponibile.")
        for model_name, run in comparison_loaded.items():
            manifest = run["manifest"]
            print(model_name, "| run:", manifest["run_id"], "| piattaforma:", manifest["platform"],
                  "| digest:", manifest["model"]["digest"])
        print("Il controllo del protocollo non garantisce ambiente e hardware identici.")


In [ ]:
if comparison_ready:
    comparison_models = list(comparison_loaded)
    comparison_metrics = [
        ("n_pairs", "Query", False),
        ("pair_success_count", "Coppie riuscite", False),
        ("paired_grounded_revision_score", "Score congiunto", True),
        ("original_grounded_score", "Supporto originale valido", True),
        ("withdrawal_score", "Ritiro corretto", True),
        ("original_answer_accuracy", "Accuracy originale", True),
        ("correct_original_without_valid_support_count", "Originali corretti senza supporto valido", False),
        ("format_errors", "Errori di formato", False),
        ("withdrawal_stale_citation_count", "Ritiri con citazioni obsolete", False),
    ]
    comparison_table = "| Misura | " + " | ".join(comparison_models) + " |\n|---|---:|---:|"
    for key, label, percentage in comparison_metrics:
        values = [run["evaluation"]["summary"][key] for run in comparison_loaded.values()]
        formatted = [f"{value:.2%}" if percentage else str(value) for value in values]
        comparison_table += "\n| " + label + " | " + " | ".join(formatted) + " |"
    display(Markdown(comparison_table))
    depth_table = "| Modello | Profondità | Query | Grounded | Ritiro | Congiunto |\n|---|---:|---:|---:|---:|---:|"
    for model_name, run in comparison_loaded.items():
        for depth, scores in sorted(run["evaluation"]["by_depth"].items(), key=lambda item: int(item[0])):
            depth_table += (f"\n| {model_name} | {depth} | {scores['n_pairs']} | "
                            f"{scores['original_grounded_score']:.2%} | {scores['withdrawal_score']:.2%} | "
                            f"{scores['paired_grounded_revision_score']:.2%} |")
    display(Markdown(depth_table))
    left, right = comparison_models
    left_pairs = {p["case_id"]: p["pair_success"] for p in comparison_loaded[left]["evaluation"]["pairs"]}
    right_pairs = {p["case_id"]: p["pair_success"] for p in comparison_loaded[right]["evaluation"]["pairs"]}
    if left_pairs.keys() != right_pairs.keys():
        raise ValueError("I run non contengono gli stessi casi.")
    display(Markdown("### Esiti sugli stessi casi"))
    for label, cases in (
        ("Entrambi superano", [c for c in left_pairs if left_pairs[c] and right_pairs[c]]),
        (f"Solo {left} supera", [c for c in left_pairs if left_pairs[c] and not right_pairs[c]]),
        (f"Solo {right} supera", [c for c in left_pairs if right_pairs[c] and not left_pairs[c]]),
        ("Nessuno supera", [c for c in left_pairs if not left_pairs[c] and not right_pairs[c]]),
    ):
        print(f"{label}: {len(cases)} — {', '.join(cases) or 'nessuno'}")
else:
    print("Tabelle dinamiche non prodotte: completare prima il caricamento e la verifica dei due run.")
